In [22]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)

llm = ChatOpenAI(
    model = "gpt-4.1-mini"
)

In [3]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name='my_documents',
    embedding_function=embeddings,
    persist_directory='./my_chroma_db'
)

### 1) Similarity Search Simple Retriever

In [14]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [15]:
query = "Who was daenerys targaryen"

results = retriever.invoke(query)

In [16]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("----------------")

THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi 
of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, 
Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL, 
-her brothers: 
-{RHAEGAR}, Prince of Dragonstone and heir to the Iron Throne, slain by King Robert on 
the Trident,
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
they made him king, he locked her up in a tower. His other sisters too. There was three.” 
“Daenela,” the proprietor said loudly. “That was her name. The Mad King’s daughter, I mean, 
not Baelor’s bloody wife.” 
“Daenerys,” Davos said. “She was named for the Daenerys who wed the Prince of Dorne during 
the reign of Daeron the Second. I don’t know what became of her.” 
“I do,” said the man who’d started all the talk of dragons, a Braa

### 2) Maximal Marginal Relevance (MMR) Retriever

In [19]:
mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'lambda_mult': 0.5}  # 'lambda_mult' : Relevance-Diversity Balance
)

In [20]:
query = "Who was daenerys targaryen"

results = mmr_retriever.invoke(query)

In [21]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("----------------")

THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi 
of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, 
Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL, 
-her brothers: 
-{RHAEGAR}, Prince of Dragonstone and heir to the Iron Throne, slain by King Robert on 
the Trident,
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
"Your Grace, the slavers brought their doom on themselves," said Daario Naharis. 
"You have brought freedom as well," Missandei pointed out. 
"Freedom to starve?" asked Dany sharply. "Freedom to die? Am I a dragon, or a harpy?" Am I mad? 
Do I have the taint? 
"A dragon," Ser Barristan said with certainty. "Meereen is not Westeros, Your Grace." 
"But how can I rule seven kingdoms if I cannot rule a single city?" He had no answer to t

### 3) Multi-Query Retriever

In [24]:
from langchain_classic.retrievers import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3}),
    llm=llm,
)

In [25]:
query = "Who was daenerys targaryen"

result = multiquery_retriever.invoke(query)

In [26]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("----------------")

THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi 
of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, 
Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL, 
-her brothers: 
-{RHAEGAR}, Prince of Dragonstone and heir to the Iron Throne, slain by King Robert on 
the Trident,
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
"Your Grace, the slavers brought their doom on themselves," said Daario Naharis. 
"You have brought freedom as well," Missandei pointed out. 
"Freedom to starve?" asked Dany sharply. "Freedom to die? Am I a dragon, or a harpy?" Am I mad? 
Do I have the taint? 
"A dragon," Ser Barristan said with certainty. "Meereen is not Westeros, Your Grace." 
"But how can I rule seven kingdoms if I cannot rule a single city?" He had no answer to t

In [27]:
query = "Who was dragon queen"

result = multiquery_retriever.invoke(query)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("----------------")

THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi 
of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, 
Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL, 
-her brothers: 
-{RHAEGAR}, Prince of Dragonstone and heir to the Iron Throne, slain by King Robert on 
the Trident,
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
"Your Grace, the slavers brought their doom on themselves," said Daario Naharis. 
"You have brought freedom as well," Missandei pointed out. 
"Freedom to starve?" asked Dany sharply. "Freedom to die? Am I a dragon, or a harpy?" Am I mad? 
Do I have the taint? 
"A dragon," Ser Barristan said with certainty. "Meereen is not Westeros, Your Grace." 
"But how can I rule seven kingdoms if I cannot rule a single city?" He had no answer to t

### 4) Contextual Compression Retriever

In [32]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


base_retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5})

In [33]:
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [34]:
query = "Who was daenerys targaryen"

results = compression_retriever.invoke(query)

In [35]:
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("----------------")

DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL,
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
"Am I a dragon, or a harpy?"
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\003ssb.txt'}
----------------
A kingly man in rich robes rose when he saw her, and smiled. "Daenerys of House Targaryen, be
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
